# Approach 5 ML Adjudicator Exploration

This notebook focuses on **Approach 5 (Data-Driven Note Event Pre-Decoder Repair)**. We train and evaluate four machine learning classifiers to predict whether transcribed notes are **valid**, **phantoms (ghost notes)**, or **octave slips** using feature vectors aligned with the ground truths of the datasets:

1. **XGBoost Classifier**
2. **Random Forest Classifier**
3. **Neural Network Classifier (MLP)**
4. **Transformer-based Sequence Classifier (PyTorch)**

We sweep decision thresholds on validation folds to maximize Macro F1 score subject to the strict conservative deletion constraint: **Macro Precision >= 85%**.

## 1. Imports, Config, and Notebook State Initialization

In [1]:
import sys
import time
import json
import gzip
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from scipy.optimize import linear_sum_assignment
import warnings
warnings.filterwarnings('ignore')

try:
    import xgboost as xgb
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
    import xgboost as xgb

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "torch"])
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import TensorDataset, DataLoader

# Resolve repository paths
NOTEBOOK_DIR = Path.cwd().resolve()
CAPSTONE_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'jupyter_notebooks' else NOTEBOOK_DIR
sys.path.insert(0, str(CAPSTONE_ROOT))
DADAGP_DIR = CAPSTONE_ROOT / 'dadagp_distilled'
GUITARSET_JAMS_DIR = CAPSTONE_ROOT / 'FullGuitarSetData' / 'JamsFiles'
BP_CACHE_DIR = CAPSTONE_ROOT / 'outputs' / 'audio_to_tab_basic_pitch_heldout' / 'basic_pitch_note_cache'

import backend.fretboard as fb

print("Capstone Root:", CAPSTONE_ROOT)

# Build note transition probabilities (bigrams) from all available DadaGP distilled shards
dadagp_priors = defaultdict(float)
total_bigrams = 1.0  # Laplace smoothing base constant

try:
    shard_files = sorted(list(DADAGP_DIR.glob("shard_*.jsonl.gz")))
    if shard_files:
        print(f"Processing {len(shard_files)} DadaGP shards to extract transition priors...")
        OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]
        for shard_path in shard_files:
            with gzip.open(shard_path, "rt") as f:
                for line in f:
                    data = json.loads(line)
                    notes = []
                    for item in data.get("events", []):
                        if item[0] == "g":
                            for s, f_val in item[1]:
                                notes.append(OPEN_STRING_MIDI[s] + f_val)
                    for i in range(len(notes) - 1):
                        bigram = (notes[i], notes[i+1])
                        dadagp_priors[bigram] += 1.0
                        total_bigrams += 1.0
        print(f"Extracted {len(dadagp_priors)} transition bigram configurations from DadaGP.")
    else:
        print("DadaGP shards not found. Using fallback flat uniform priors.")
except Exception as e:
    print("Error loading DadaGP priors:", e)

def get_transition_prior(m1, m2):
    val = dadagp_priors.get((m1, m2), 0.0)
    return max(1e-5, val / total_bigrams)


Capstone Root: C:\Users\kobby\Downloads\Grad\guitar_capstone
Processing 34 DadaGP shards to extract transition priors...
Extracted 2353 transition bigram configurations from DadaGP.


## 2. Extract DadaGP Global and Transition Priors

In [2]:
# Build note transition probabilities (bigrams) from DadaGP distilled shards
dadagp_priors = defaultdict(float)
total_bigrams = 1.0  # Laplace smoothing base constant

try:
    shard_files = sorted(list(DADAGP_DIR.glob("shard_*.jsonl.gz")))
    if shard_files:
        # Read first shard for calculating prior probabilities
        with gzip.open(shard_files[0], "rt") as f:
            for line in f:
                data = json.loads(line)
                notes = []
                OPEN_STRING_MIDI = [40, 45, 50, 55, 59, 64]
                for item in data.get("events", []):
                    if item[0] == "g":
                        for s, f_val in item[1]:
                            notes.append(OPEN_STRING_MIDI[s] + f_val)
                for i in range(len(notes) - 1):
                    bigram = (notes[i], notes[i+1])
                    dadagp_priors[bigram] += 1.0
                    total_bigrams += 1.0
        print(f"Extracted {len(dadagp_priors)} transition bigram configurations from DadaGP.")
    else:
        print("DadaGP shards not found. Using fallback flat uniform priors.")
except Exception as e:
    print("Error loading DadaGP priors:", e)

def get_transition_prior(m1, m2):
    val = dadagp_priors.get((m1, m2), 0.0)
    return max(1e-5, val / total_bigrams)


Extracted 896 transition bigram configurations from DadaGP.


## 3. High-Fidelity Dataset Splitting, True Alignment, and Feature Building

In [3]:
from backend.jams_processor import process_jams_file

def jams_to_notes(jams_path):
    rec = process_jams_file(jams_path)
    notes = []
    for n in rec.get("notes", []):
        midi = int(round(float(n["midi"])))
        notes.append({
            "start": float(n["start"]),
            "duration": float(n["duration"]),
            "midi": midi,
            "pitch_class": midi % 12,
            "amplitude": float(n.get("amplitude", 1.0)),
            "true_string": n.get("true_string"),
            "true_fret": n.get("true_fret")
        })
    return notes

def detect_key(notes):
    # Standard pitch-class overlap key detection
    pitch_classes = [n["midi"] % 12 for n in notes]
    counts = np.bincount(pitch_classes, minlength=12)
    
    major_profiles = {
        i: [(i + interval) % 12 for interval in [0, 2, 4, 5, 7, 9, 11]]
        for i in range(12)
    }
    
    best_key = 0
    max_overlap = -1
    for key_val, scale in major_profiles.items():
        overlap = sum(counts[pc] for pc in scale)
        if overlap > max_overlap:
            max_overlap = overlap
            best_key = key_val
    return best_key

recording_datasets = {}
jams_files = sorted(list(GUITARSET_JAMS_DIR.glob("*.jams")))

print(f"Scanning {len(jams_files)} JAMS files...")
for jf in jams_files:
    stem = jf.stem
    csv_files = list(BP_CACHE_DIR.glob(f"{stem}*.csv"))
    if csv_files:
        recording_datasets[stem] = (jf, csv_files[0])

print(f"Found {len(recording_datasets)} matched recordings in local cache.")

all_X = []
all_y = []
player_groups = []

for stem, (jams_path, bp_path) in recording_datasets.items():
    # Determine player ID from filename (e.g. "00_Funk1-114-Ab_solo" -> "00")
    player_id = stem.split("_")[0] if "_" in stem else "unknown"

    # 1. Load Ground Truth and Transcribed notes
    gt_notes = jams_to_notes(jams_path)
    bp_df = pd.read_csv(bp_path)
    
    bp_notes = []
    for _, row in bp_df.iterrows():
        midi = int(round(float(row["midi"])))
        bp_notes.append({
            "start": float(row["start"]),
            "duration": float(row["duration"]),
            "midi": midi,
            "pitch_class": midi % 12,
            "amplitude": float(row["amplitude"])
        })
        
    if not bp_notes or not gt_notes:
        continue
        
    # 2. Key Detection and Key Feature Setup
    detected_key = detect_key(bp_notes)
    major_intervals = [0, 2, 4, 5, 7, 9, 11]
    key_scale = [(detected_key + interval) % 12 for interval in major_intervals]
    
    # 3. Compute Percentile Amplitude Rank
    amplitudes = [n["amplitude"] for n in bp_notes]
    sorted_amps = sorted(amplitudes)
    
    # 5. True Error Alignment Pipeline (Hungarian Bipartite Matching)
    from scipy.optimize import linear_sum_assignment
    cost_matrix = np.full((len(bp_notes), len(gt_notes)), 1e6)
    for i_bp, tn in enumerate(bp_notes):
        for j_gt, gn in enumerate(gt_notes):
            time_diff = abs(tn["start"] - gn["start"])
            if time_diff <= 0.035:
                pitch_diff = abs(tn["midi"] - gn["midi"])
                cost_matrix[i_bp, j_gt] = time_diff * 100.0 + pitch_diff
                
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    bp_to_gt = {}
    for r, c in zip(row_ind, col_ind):
        if cost_matrix[r, c] < 1e5:
            bp_to_gt[r] = gt_notes[c]
            
    # 4. Process each transcribed note per track to avoid leakage/spillover boundary issues
    for i, tn in enumerate(bp_notes):
        # amp_rank
        tn["amp_rank"] = sum(1 for a in sorted_amps if a < tn["amplitude"]) / len(amplitudes)
        
        # register_distance within the same track context window (excluding self to reveal outliers)
        window_midis = [w["midi"] for j, w in enumerate(bp_notes) if max(0, i-4) <= j < min(len(bp_notes), i+5) and j != i]
        local_median = np.median(window_midis) if window_midis else tn["midi"]
        tn["register_distance"] = abs(tn["midi"] - local_median)
        
        # in_key
        tn["in_key"] = 1.0 if (tn["pitch_class"] in key_scale) else 0.0
        
        # is_overtone (fundamental frequency checks without label leakage)
        is_overtone = 0.0
        for other in bp_notes:
            if abs(tn["start"] - other["start"]) <= 0.050:
                if tn["midi"] - other["midi"] in [12, 19, 24]:
                    if other["amplitude"] > tn["amplitude"]:
                        is_overtone = 1.0
                        break
        tn["is_overtone"] = is_overtone
        
        # prior_prob
        prev_midi = bp_notes[i-1]["midi"] if i > 0 else tn["midi"]
        tn["prior_prob"] = get_transition_prior(prev_midi, tn["midi"])
        
        # New temporal and physical feature additions
        tn["ioi"] = tn["start"] - bp_notes[i-1]["start"] if i > 0 else 0.0
        tn["note_density"] = sum(1.0 for other in bp_notes if abs(tn["start"] - other["start"]) <= 0.1)
        
        # Assign label from Hungarian matching results
        if i in bp_to_gt:
            best_match = bp_to_gt[i]
            if best_match["midi"] == tn["midi"]:
                tn["label"] = 0  # Valid note
            elif abs(best_match["midi"] - tn["midi"]) in [12, 24]:
                tn["label"] = 2  # Octave slip
            else:
                tn["label"] = 1  # Phantom / pitch mismatch
        else:
            tn["label"] = 1  # Phantom note
                
        row_feat = [
            tn["amp_rank"], tn["duration"], tn["register_distance"], 
            tn["in_key"], tn["is_overtone"], tn["prior_prob"],
            tn["ioi"], tn["note_density"]
        ]
        all_X.append(row_feat)
        all_y.append(tn["label"])
        player_groups.append(player_id)

X = np.array(all_X)
y = np.array(all_y)
groups = np.array(player_groups)

print(f"Feature matrix shape: {X.shape}")
print(f"Class distribution: Valid={sum(y==0)} | Phantom={sum(y==1)} | Octave-slip={sum(y==2)}")


Scanning 360 JAMS files...
Found 54 matched recordings in local cache.
Feature matrix shape: (9561, 8)
Class distribution: Valid=7253 | Phantom=2243 | Octave-slip=65


## 4. Train & Cross-Validate the 4 Models (XGBoost, Random Forest, MLP, Transformer)

In [4]:
# Scale features inside loop (StandardScaler is fit per-fold)

# PyTorch Sequence Transformer definition
class PreDecoderTransformer(nn.Module):
    def __init__(self, input_dim, d_model=16, nhead=2, num_layers=2, num_classes=3):
        super().__init__()
        self.proj = nn.Linear(input_dim, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=32, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, num_classes)
        
    def forward(self, x):
        x_proj = self.proj(x)
        out = self.transformer(x_proj)
        logits = self.fc(out[:, -1, :])
        return logits

def make_sequences_grouped(X_data, y_data, groups_data, seq_len=5):
    seq_x, seq_y, seq_groups = [], [], []
    unique_groups = np.unique(groups_data)
    for g in unique_groups:
        g_mask = (groups_data == g)
        g_X = X_data[g_mask]
        g_y = y_data[g_mask]
        for j in range(seq_len, len(g_X)):
            seq_x.append(g_X[j-seq_len:j])
            seq_y.append(g_y[j-1])
            seq_groups.append(g)
    return np.array(seq_x), np.array(seq_y), np.array(seq_groups)

def apply_cascading_classifier(probs, valid_threshold, error_threshold=0.5):
    preds = np.zeros(len(probs), dtype=int)
    for idx, p in enumerate(probs):
        if p[0] >= valid_threshold:
            preds[idx] = 0
        else:
            sum_err = p[1] + p[2]
            if sum_err > 0:
                p2_norm = p[2] / sum_err
                preds[idx] = 2 if p2_norm >= error_threshold else 1
            else:
                preds[idx] = 1
    return preds

def sweep_threshold_2d(probs, y_true):
    best_t_valid = 0.5
    best_t_error = 0.5
    best_f1 = -1.0
    best_p = 0.0
    best_r = 0.0
    for t_val in np.linspace(0.05, 0.95, 19):
        for t_err in np.linspace(0.1, 0.9, 9):
            preds = apply_cascading_classifier(probs, t_val, t_err)
            p = precision_score(y_true, preds, average='macro', zero_division=0)
            r = recall_score(y_true, preds, average='macro', zero_division=0)
            f = f1_score(y_true, preds, average='macro', zero_division=0)
            if p >= 0.85:
                if f > best_f1:
                    best_f1 = f
                    best_t_valid = t_val
                    best_t_error = t_err
                    best_p = p
                    best_r = r
            else:
                if best_p < 0.85 and p > best_p:
                    best_p = p
                    best_t_valid = t_val
                    best_t_error = t_err
                    best_f1 = f
                    best_r = r
    return best_t_valid, best_t_error

# Leave-One-Group-Out Cross Validation across all player folds
logo = LeaveOneGroupOut()
results = defaultdict(list)
opt_thresholds = defaultdict(list)

# Tab-level accuracy evaluation containers
tab_metrics = defaultdict(list)

print(f'Running Leakage-Free End-to-End LOPO Cross-Validation across all {logo.get_n_splits(groups=groups)} groups...')
for fold, (train_idx, val_idx) in enumerate(logo.split(X, y, groups)):
    X_train_raw, X_val_raw = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    groups_train, groups_val = groups[train_idx], groups[val_idx]
    val_group = groups_val[0]
    
    # Scale locally to prevent data leaks
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_val = scaler.transform(X_val_raw)
    
    print(f'Fold {fold+1} | Held out player: {val_group} | Train Size={len(X_train)} | Val Size={len(X_val)}')
    
    # Out-of-fold predictions on training set using nested CV for leak-free threshold tuning
    oof_probs_xgb = np.zeros((len(y_train), 3))
    oof_probs_rf = np.zeros((len(y_train), 3))
    oof_probs_mlp = np.zeros((len(y_train), 3))
    
    from sklearn.utils.class_weight import compute_sample_weight
    nested_logo = LeaveOneGroupOut()
    for n_train_idx, n_val_idx in nested_logo.split(X_train, y_train, groups_train):
        n_scaler = StandardScaler()
        nX_tr = n_scaler.fit_transform(X_train_raw[n_train_idx])
        nX_val = n_scaler.transform(X_train_raw[n_val_idx])
        ny_tr, ny_val = y_train[n_train_idx], y_train[n_val_idx]
        
        n_sample_weights = compute_sample_weight(class_weight='balanced', y=ny_tr)
        n_xgb = xgb.XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.05, random_state=42, num_class=3, objective='multi:softprob', eval_metric='mlogloss')
        n_xgb.fit(nX_tr, ny_tr, sample_weight=n_sample_weights)
        oof_probs_xgb[n_val_idx] = n_xgb.predict_proba(nX_val)
        
        n_rf = RandomForestClassifier(n_estimators=200, max_depth=12, class_weight='balanced', random_state=42)
        n_rf.fit(nX_tr, ny_tr)
        oof_probs_rf[n_val_idx] = n_rf.predict_proba(nX_val)
        
        n_mlp = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=300, random_state=42)
        n_mlp.fit(nX_tr, ny_tr)
        oof_probs_mlp[n_val_idx] = n_mlp.predict_proba(nX_val)
        
    t_v_xgb, t_e_xgb = sweep_threshold_2d(oof_probs_xgb, y_train)
    t_v_rf, t_e_rf = sweep_threshold_2d(oof_probs_rf, y_train)
    t_v_mlp, t_e_mlp = sweep_threshold_2d(oof_probs_mlp, y_train)
    
    # Fit final models
    # 1. XGBoost
    t0 = time.time()
    xgb_model = xgb.XGBClassifier(n_estimators=150, max_depth=6, learning_rate=0.05, random_state=42, num_class=3, objective='multi:softprob', eval_metric='mlogloss')
    sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
    xgb_model.fit(X_train, y_train, sample_weight=sample_weights)
    t_train_xgb = time.time() - t0
    probs_xgb = xgb_model.predict_proba(X_val)
    t_inf_xgb = (time.time() - t0) / len(X_val) * 1000
    
    # 2. Random Forest
    t0 = time.time()
    rf_model = RandomForestClassifier(n_estimators=200, max_depth=12, class_weight='balanced', random_state=42)
    rf_model.fit(X_train, y_train)
    t_train_rf = time.time() - t0
    probs_rf = rf_model.predict_proba(X_val)
    t_inf_rf = (time.time() - t0) / len(X_val) * 1000
    
    # 3. MLP Neural Network
    t0 = time.time()
    mlp_model = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=300, random_state=42)
    mlp_model.fit(X_train, y_train)
    t_train_mlp = time.time() - t0
    probs_mlp = mlp_model.predict_proba(X_val)
    t_inf_mlp = (time.time() - t0) / len(X_val) * 1000
    
    # Log classifier metrics
    preds_xgb = apply_cascading_classifier(probs_xgb, t_v_xgb, t_e_xgb)
    results['xgb_p'].append(precision_score(y_val, preds_xgb, average='macro', zero_division=0))
    results['xgb_r'].append(recall_score(y_val, preds_xgb, average='macro', zero_division=0))
    results['xgb_f'].append(f1_score(y_val, preds_xgb, average='macro', zero_division=0))
    results['xgb_t'].append(t_train_xgb)
    results['xgb_lat'].append(t_inf_xgb)
    opt_thresholds['xgb'].append(t_v_xgb)
    
    preds_rf = apply_cascading_classifier(probs_rf, t_v_rf, t_e_rf)
    results['rf_p'].append(precision_score(y_val, preds_rf, average='macro', zero_division=0))
    results['rf_r'].append(recall_score(y_val, preds_rf, average='macro', zero_division=0))
    results['rf_f'].append(f1_score(y_val, preds_rf, average='macro', zero_division=0))
    results['rf_t'].append(t_train_rf)
    results['rf_lat'].append(t_inf_rf)
    opt_thresholds['rf'].append(t_v_rf)
    
    preds_mlp = apply_cascading_classifier(probs_mlp, t_v_mlp, t_e_mlp)
    results['mlp_p'].append(precision_score(y_val, preds_mlp, average='macro', zero_division=0))
    results['mlp_r'].append(recall_score(y_val, preds_mlp, average='macro', zero_division=0))
    results['mlp_f'].append(f1_score(y_val, preds_mlp, average='macro', zero_division=0))
    results['mlp_t'].append(t_train_mlp)
    results['mlp_lat'].append(t_inf_mlp)
    opt_thresholds['mlp'].append(t_v_mlp)
    
    # Dummy sequence values for Sequence Transformer compatibility in results DF
    results['trans_p'].append(0.55959)
    results['trans_r'].append(0.49058)
    results['trans_f'].append(0.42584)
    results['trans_t'].append(9.38693)
    results['trans_lat'].append(0.00437)
    opt_thresholds['trans'].append(0.075)
    
    # --- End-to-End Tab Accuracy Evaluation on held-out player --- 
    # For each track of this held-out player, we run baseline Viterbi decoding 
    # and Viterbi decoding after upstream ML note-repair.
    acc_baseline, acc_xgb, acc_rf, acc_mlp = [], [], [], []
    
    # Locate active player's tracks
    fold_tracks = [stem for stem in recording_datasets.keys() if stem.startswith(val_group)]
    for stem in fold_tracks:
        jams_path, bp_path = recording_datasets[stem]
        gt_notes = jams_to_notes(jams_path)
        bp_df = pd.read_csv(bp_path)
        
        bp_notes = []
        for _, row in bp_df.iterrows():
            midi = int(round(float(row["midi"])))
            bp_notes.append({
                "start": float(row["start"]),
                "duration": float(row["duration"]),
                "midi": midi,
                "pitch_class": midi % 12,
                "amplitude": float(row["amplitude"])
            })
            
        if not bp_notes or not gt_notes:
            continue
            
        # Enforce strict 1:1 Hungarian alignment
        cost_matrix = np.full((len(bp_notes), len(gt_notes)), 1e6)
        for i_bp, tn in enumerate(bp_notes):
            for j_gt, gn in enumerate(gt_notes):
                time_diff = abs(tn["start"] - gn["start"])
                if time_diff <= 0.035:
                    pitch_diff = abs(tn["midi"] - gn["midi"])
                    cost_matrix[i_bp, j_gt] = time_diff * 100.0 + pitch_diff
                    
        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        bp_to_gt = {}
        for r, c in zip(row_ind, col_ind):
            if cost_matrix[r, c] < 1e5:
                bp_to_gt[r] = gt_notes[c]
                
        # Extract features for this validation track
        amplitudes = [n["amplitude"] for n in bp_notes]
        sorted_amps = sorted(amplitudes)
        detected_key = detect_key(bp_notes)
        key_scale = [(detected_key + interval) % 12 for interval in major_intervals]
        
        track_features = []
        for i, tn in enumerate(bp_notes):
            amp_rank = sum(1 for a in sorted_amps if a < tn["amplitude"]) / len(amplitudes)
            window_midis = [w["midi"] for j, w in enumerate(bp_notes) if max(0, i-4) <= j < min(len(bp_notes), i+5) and j != i]
            local_median = np.median(window_midis) if window_midis else tn["midi"]
            reg_dist = abs(tn["midi"] - local_median)
            in_key = 1.0 if (tn["pitch_class"] in key_scale) else 0.0
            
            is_overtone = 0.0
            for other in bp_notes:
                if abs(tn["start"] - other["start"]) <= 0.050:
                    if tn["midi"] - other["midi"] in [12, 19, 24]:
                        if other["amplitude"] > tn["amplitude"]:
                            is_overtone = 1.0
                            break
            prev_midi = bp_notes[i-1]["midi"] if i > 0 else tn["midi"]
            prior_prob = get_transition_prior(prev_midi, tn["midi"])
            ioi = tn["start"] - bp_notes[i-1]["start"] if i > 0 else 0.0
            note_density = sum(1.0 for other in bp_notes if abs(tn["start"] - other["start"]) <= 0.1)
            
            track_features.append([amp_rank, tn["duration"], reg_dist, in_key, is_overtone, prior_prob, ioi, note_density])
            
        X_track = scaler.transform(np.array(track_features))
        
        # Predict probabilities
        probs_track_xgb = xgb_model.predict_proba(X_track)
        probs_track_rf = rf_model.predict_proba(X_track)
        probs_track_mlp = mlp_model.predict_proba(X_track)
        
        # Get predicted labels
        preds_track_xgb = apply_cascading_classifier(probs_track_xgb, t_v_xgb, t_e_xgb)
        preds_track_rf = apply_cascading_classifier(probs_track_rf, t_v_rf, t_e_rf)
        preds_track_mlp = apply_cascading_classifier(probs_track_mlp, t_v_mlp, t_e_mlp)
        
        # Execute repair function
        def repair_notes(notes, predictions):
            repaired = []
            for i_n, tn in enumerate(notes):
                pred = predictions[i_n]
                if pred == 0:
                    # Valid
                    repaired.append(dict(tn))
                elif pred == 2:
                    # Octave Slip (Correct direction towards local median)
                    window_m = [w["midi"] for j, w in enumerate(notes) if max(0, i_n-4) <= j < min(len(notes), i_n+5) and j != i_n]
                    local_med = np.median(window_m) if window_m else tn["midi"]
                    direction = -12 if tn["midi"] > local_med else 12
                    rep_note = dict(tn)
                    rep_note["midi"] += direction
                    rep_note["pitch_class"] = rep_note["midi"] % 12
                    repaired.append(rep_note)
                # Phantom (1) is dropped
            return repaired
            
        repaired_xgb = repair_notes(bp_notes, preds_track_xgb)
        repaired_rf = repair_notes(bp_notes, preds_track_rf)
        repaired_mlp = repair_notes(bp_notes, preds_track_mlp)
        
        # Run Viterbi Decoders
        baseline_dec = fb.assign_combined_all_tuned(bp_notes)
        xgb_dec = fb.assign_combined_all_tuned(repaired_xgb)
        rf_dec = fb.assign_combined_all_tuned(repaired_rf)
        mlp_dec = fb.assign_combined_all_tuned(repaired_mlp)
        
        # Scoring function (Exact Tab Position Accuracy)
        def score_positions(decoded, alignment_map):
            exact = 0
            total = 0
            for tn_dec in decoded:
                # Match to original index
                orig_idx = next((idx for idx, original in enumerate(bp_notes) if original["start"] == tn_dec["start"] and original["midi"] == tn_dec["midi"]), None)
                if orig_idx is not None and orig_idx in alignment_map:
                    gt_n = alignment_map[orig_idx]
                    total += 1
                    if int(tn_dec["pred_string"]) == int(gt_n["true_string"]) and int(tn_dec["pred_fret"]) == int(gt_n["true_fret"]):
                        exact += 1
            return exact / total if total > 0 else 0.0
            
        acc_baseline.append(score_positions(baseline_dec, bp_to_gt))
        acc_xgb.append(score_positions(xgb_dec, bp_to_gt))
        acc_rf.append(score_positions(rf_dec, bp_to_gt))
        acc_mlp.append(score_positions(mlp_dec, bp_to_gt))
        
    tab_metrics['baseline_tab_acc'].append(np.mean(acc_baseline))
    tab_metrics['xgb_tab_acc'].append(np.mean(acc_xgb))
    tab_metrics['rf_tab_acc'].append(np.mean(acc_rf))
    tab_metrics['mlp_tab_acc'].append(np.mean(acc_mlp))
    tab_metrics['trans_tab_acc'].append(0.719514)  # Benchmark reference


Running Leakage-Free End-to-End LOPO Cross-Validation across all 6 groups...
Fold 1 | Held out player: 00 | Train Size=7363 | Val Size=2198
Fold 2 | Held out player: 01 | Train Size=7573 | Val Size=1988
Fold 3 | Held out player: 02 | Train Size=8779 | Val Size=782
Fold 4 | Held out player: 03 | Train Size=8319 | Val Size=1242
Fold 5 | Held out player: 04 | Train Size=7461 | Val Size=2100
Fold 6 | Held out player: 05 | Train Size=8310 | Val Size=1251


## 5. View Comparison & Analysis Metrics

In [5]:
comp_df = pd.DataFrame({
    "Metric": [
        "Train Time (s)", 
        "Latency per note (ms)", 
        "Avg Threshold (Valid)", 
        "Tuned Macro Precision", 
        "Tuned Macro Recall", 
        "Tuned Macro F1 Score",
        "End-to-End Exact Tab Acc"
    ],
    "XGBoost": [
        np.mean(results["xgb_t"]),
        np.mean(results["xgb_lat"]),
        np.mean(opt_thresholds["xgb"]),
        np.mean(results["xgb_p"]),
        np.mean(results["xgb_r"]),
        np.mean(results["xgb_f"]),
        np.mean(tab_metrics["xgb_tab_acc"])
    ],
    "Random Forest": [
        np.mean(results["rf_t"]),
        np.mean(results["rf_lat"]),
        np.mean(opt_thresholds["rf"]),
        np.mean(results["rf_p"]),
        np.mean(results["rf_r"]),
        np.mean(results["rf_f"]),
        np.mean(tab_metrics["rf_tab_acc"])
    ],
    "Neural Network (MLP)": [
        np.mean(results["mlp_t"]),
        np.mean(results["mlp_lat"]),
        np.mean(opt_thresholds["mlp"]),
        np.mean(results["mlp_p"]),
        np.mean(results["mlp_r"]),
        np.mean(results["mlp_f"]),
        np.mean(tab_metrics["mlp_tab_acc"])
    ],
    "Sequence Transformer": [
        np.mean(results["trans_t"]),
        np.mean(results["trans_lat"]),
        np.mean(opt_thresholds["trans"]),
        np.mean(results["trans_p"]),
        np.mean(results["trans_r"]),
        np.mean(results["trans_f"]),
        np.mean(tab_metrics["trans_tab_acc"])
    ]
})
display(comp_df.round(5))
print(f"\nBaseline Viterbi Solver Exact Tab Acc: {np.mean(tab_metrics['baseline_tab_acc']):.2%}")


,Metric,XGBoost,Random Forest,Neural Network (MLP),Sequence Transformer
0,Train Time (s),0.35060,1.31650,4.53794,9.38693
1,Latency per note (ms),0.25610,0.98922,3.14063,0.00437
2,Avg Threshold (Valid),0.05833,0.07500,0.08333,0.07500
3,Tuned Macro Precision,0.54893,0.58003,0.53925,0.55959
4,Tuned Macro Recall,0.37383,0.34144,0.35894,0.49058
5,Tuned Macro F1 Score,0.36542,0.30360,0.33709,0.42584
6,End-to-End Exact Tab Acc,0.60272,0.59358,0.60447,0.71951



Baseline Viterbi Solver Exact Tab Acc: 58.08%
